# 04d - Secondary Python Logistic Validation Model

Short summary:
[04d - Secondary Python Logistic Validation Model (SHORT)](../docs/04d_python_logistic_model_SHORT.md)

---

## 1. Objective
Fit the Python logistic regression model used to generate parameter estimates for SAS-Python reproducibility validation.

---

## 2. Data Source
**Input Dataset**
- `../sas/inputs/copd_raas__cohort_copd_outcomes_extended.csv`

**Output Dataset**
- `../python/outputs/python_logistic_parameters.csv`

**Downstream Usage**
- The exported parameter file is used by `05_sas_python_validation.ipynb` for SAS/Python reproducibility validation.

The model uses the exported COPD RAAS cohort dataset from the project data pipeline and writes Python parameter estimates for validation in `05_sas_python_validation.ipynb`.

---

## 3. Study Design
This notebook uses the exported analysis cohort only to reproduce a secondary binary-outcome logistic specification for SAS-Python validation; it is not the primary clinical model.

---

## 4. Statistical Methods
A SAS-equivalent logistic regression specification is fitted in Python, and parameter estimates are exported for downstream validation.

---

## 5. Results
### 5.1 Load Cohort Dataset


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

input_path = Path("../sas/inputs/copd_raas__cohort_copd_outcomes_extended.csv")
output_path = Path("../python/outputs/python_logistic_parameters.csv")

df = pd.read_csv(input_path)
df.shape


### 5.2 Fit SAS-equivalent Logistic Regression


In [ ]:
model_columns = [
    "death_event",
    "raas_pre_icu",
    "age",
    "gender",
    "sofa_score",
    "charlson_comorbidity_index",
    "chf",
    "ckd",
    "diabetes",
]

model_df = df[model_columns].dropna().copy()
model_df["gender_M"] = (model_df["gender"] == "M").astype(int)

y = model_df["death_event"].astype(float)
X = model_df[[
    "raas_pre_icu",
    "age",
    "gender_M",
    "sofa_score",
    "charlson_comorbidity_index",
    "chf",
    "ckd",
    "diabetes",
]].astype(float)
X = sm.add_constant(X, has_constant="add")

logit_result = sm.Logit(y, X).fit()
logit_result.summary()


### 5.3 Export Parameter Results


The exported CSV is generated locally and is not committed to the repository. The displayed table below documents the model output used for validation.


In [ ]:
params = logit_result.params
conf_int = logit_result.conf_int()

python_logistic_parameters = pd.DataFrame({
    "Variable": params.index.astype(str),
    "Python_Estimate": params.values,
    "Python_OR": np.exp(params.values),
    "Python_pvalue": logit_result.pvalues.reindex(params.index).values,
    "Python_CI_Lower": np.exp(conf_int.iloc[:, 0].reindex(params.index).values),
    "Python_CI_Upper": np.exp(conf_int.iloc[:, 1].reindex(params.index).values),
})

python_logistic_parameters["Variable"] = python_logistic_parameters["Variable"].replace({
    "const": "Intercept",
    "gender_M": "gender",
    "charlson_comorbidity_index": "charlson_comorbidity",
})

output_path.parent.mkdir(parents=True, exist_ok=True)
python_logistic_parameters.to_csv(output_path, index=False)

python_logistic_parameters.loc[
    python_logistic_parameters["Variable"].isin(["raas_pre_icu", "Intercept"])
]


---

## 6. Discussion
The exported Python model parameters provide an independent implementation of the logistic regression specification for comparison with SAS output in `05_sas_python_validation.ipynb`.

---

## 7. Conclusion
The notebook produces the Python parameter file required for reproducibility validation.


----

## Next Step

Proceed to:
- [05 - SAS-Python Validation](05_sas_python_validation.ipynb)

This next notebook compares exported SAS and Python parameter estimates for reproducibility.
